# Biohub model43 — gap closing plus real-division branch

Checksum-pinned primary temporal UNet, four-view XY TTA, ILP, broadly stable internal gap closing, and frozen real-label division geometry repair.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

import torch

COMPETITION = "biohub-cell-tracking-during-development"
EXPECTED_WEIGHT_SHA256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
EXPECTED_PREDICTOR_SHA256 = "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9"
DET_THRESHOLD = 0.965
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.0
ILP_DISAPPEARANCE_WEIGHT = 2.0
ILP_DIVISION_WEIGHT = 1.2
GAP_RADIUS_GRID = 5.0
GAP_MIN_ACCELERATION_UM = 6.5

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
COMPETITION_CANDIDATES = [
    INPUT_ROOT / "competitions" / COMPETITION,
    INPUT_ROOT / COMPETITION,
]
COMPETITION_DIR = next((path for path in COMPETITION_CANDIDATES if path.is_dir()), None)
if COMPETITION_DIR is None:
    raise FileNotFoundError({"competition_candidates": [str(path) for path in COMPETITION_CANDIDATES]})
TEST_DIR = COMPETITION_DIR / "test"
test_stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
if not test_stems:
    raise RuntimeError(f"No test Zarrs found under {TEST_DIR}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; launch this notebook with --accelerator NvidiaTeslaT4")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT != 2:
    raise RuntimeError(f"Expected Kaggle T4 x2, found {GPU_COUNT} CUDA device(s)")
GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]
if not all("T4" in name for name in GPU_NAMES):
    raise RuntimeError(f"Expected Tesla T4 devices, found {GPU_NAMES}")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def artifact_weight_sha256(root: Path) -> str:
    manifest_path = root / "ARTIFACT_MANIFEST.json"
    if not manifest_path.is_file():
        return ""
    try:
        payload = json.loads(manifest_path.read_text())
    except Exception:
        return ""
    direct = str(payload.get("model", {}).get("weight_sha256", ""))
    nested = str(
        payload.get("models", {})
        .get("unet_transformer", {})
        .get("weight_sha256", "")
    )
    return direct or nested


def candidate_artifact_roots() -> list[Path]:
    preferred = [
        INPUT_ROOT / "biohub-tracking-support-pack-50ep-v1",
        INPUT_ROOT / "datasets" / "pilkwang" / "biohub-tracking-support-pack-50ep-v1",
    ]
    discovered: list[Path] = []
    if INPUT_ROOT.is_dir():
        for child in INPUT_ROOT.iterdir():
            if not child.is_dir():
                continue
            discovered.extend([child, child / child.name])
            try:
                discovered.extend(path for path in child.iterdir() if path.is_dir())
            except OSError:
                pass
    output: list[Path] = []
    seen: set[Path] = set()
    for path in [*preferred, *discovered]:
        if path in seen:
            continue
        seen.add(path)
        output.append(path)
    return output


ARTIFACTS = next(
    (
        root
        for root in candidate_artifact_roots()
        if artifact_weight_sha256(root) == EXPECTED_WEIGHT_SHA256
        and (root / "repo" / "scripts" / "predict_unet_transformer.py").is_file()
        and (root / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth").is_file()
    ),
    None,
)
if ARTIFACTS is None:
    raise FileNotFoundError(
        "Could not find the checksum-pinned primary support pack; attach "
        "pilkwang/biohub-tracking-support-pack-50ep-v1"
    )

# Install only packages absent or known to be incompatible.  NumPy, SciPy,
# and torch are intentionally left to Kaggle's coherent GPU image.
PACKAGE_SPECS = [
    "tracksdata", "zarr>=3.0.10,<4", "numcodecs>=0.13,<0.16", "blosc2",
    "geff>=1.1.3.1.1", "geff-spec<1.2", "ilpy>=0.5.1", "pyscipopt",
    "polars>=1.36", "polars-runtime-32", "dask", "imagecodecs", "pyarrow",
    "rustworkx>=0.17.1", "sqlalchemy>=2", "bidict", "psygnal", "pydantic",
    "pydantic-core", "annotated-types", "typing-inspection", "rich",
    "markdown-it-py", "pygments", "donfig", "google-crc32c", "networkx",
    "deprecated", "wrapt", "ndindex", "msgpack", "numexpr", "click",
    "cloudpickle", "fsspec", "partd", "locket", "toolz", "pyyaml",
    "scikit-image", "imageio", "pillow", "tifffile", "lazy-loader", "tqdm",
]
REQUIRED_MODULES = [
    "tracksdata", "zarr", "numcodecs", "blosc2", "geff", "geff_spec",
    "ilpy", "pyscipopt", "polars", "dask", "imagecodecs", "pyarrow",
    "rustworkx", "sqlalchemy", "skimage", "tqdm",
]


def dependencies_ready() -> bool:
    if any(importlib.util.find_spec(name) is None for name in REQUIRED_MODULES):
        return False
    try:
        # Inspect package metadata without importing native extensions.  If an
        # old Kaggle image needs the bundled upgrade, this avoids retaining a
        # stale polars/zarr module in sys.modules after pip replaces it.
        zarr_major = int(importlib.metadata.version("zarr").split(".", 1)[0])
        polars_parts = tuple(
            int(part) for part in importlib.metadata.version("polars").split(".")[:2]
        )
        return zarr_major >= 3 and polars_parts >= (1, 36)
    except Exception:
        return False


if not dependencies_ready():
    wheel_dir = ARTIFACTS / "wheels"
    if not wheel_dir.is_dir():
        raise FileNotFoundError(f"Offline wheel directory missing: {wheel_dir}")
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--find-links",
        str(wheel_dir),
        *PACKAGE_SPECS,
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout or "")[-3000:])
    print((result.stderr or "")[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    importlib.invalidate_caches()
if not dependencies_ready():
    failures = {}
    for module_name in REQUIRED_MODULES:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[module_name] = f"{type(exc).__name__}: {exc}"
    raise ImportError({"dependency_failures": failures})

REPO_DIR = WORKING_DIR / "model43_repo"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR)
weights_destination = REPO_DIR / "weights"
try:
    os.symlink(ARTIFACTS / "weights", weights_destination, target_is_directory=True)
except Exception:
    shutil.copytree(ARTIFACTS / "weights", weights_destination)

PREDICTOR_PATH = REPO_DIR / "scripts" / "predict_unet_transformer.py"
WEIGHT_PATH = REPO_DIR / "weights" / "unet_transformer" / "split_0" / "edge_predictor_best.pth"
actual_predictor_sha256 = sha256_file(PREDICTOR_PATH)
actual_weight_sha256 = sha256_file(WEIGHT_PATH)
if actual_predictor_sha256 != EXPECTED_PREDICTOR_SHA256:
    raise RuntimeError({"predictor_sha256": actual_predictor_sha256})
if actual_weight_sha256 != EXPECTED_WEIGHT_SHA256:
    raise RuntimeError({"weight_sha256": actual_weight_sha256})

print(json.dumps({
    "artifact_root": str(ARTIFACTS),
    "competition_dir": str(COMPETITION_DIR),
    "test_datasets": len(test_stems),
    "gpu_names": GPU_NAMES,
    "predictor_sha256": actual_predictor_sha256,
    "weight_sha256": actual_weight_sha256,
}, indent=2))


In [ ]:
split_path = REPO_DIR / "model43_test_split.json"
split_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2) + "\n")

base_command = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR),
    "--splits", str(split_path),
    "--split", "0",
    "--weights", "weights/unet_transformer/split_0/edge_predictor_best.pth",
    "--unet-batch-size", "4",
    "--det-threshold", str(DET_THRESHOLD),
    "--use-ilp",
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
]

started = time.monotonic()
processes: list[tuple[list[str], subprocess.Popen]] = []
for gpu_index in range(GPU_COUNT):
    method = f"model43_primary_gpu{gpu_index}"
    command = [*base_command, "--method", method, "--slice", f"{gpu_index}::{GPU_COUNT}"]
    environment = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": str(gpu_index),
        "PYTHONPATH": "src",
        "OMP_NUM_THREADS": "2",
    }
    print("Launching:", " ".join(command), flush=True)
    processes.append((command, subprocess.Popen(command, cwd=REPO_DIR, env=environment)))

failures = []
for command, process in processes:
    returncode = process.wait()
    if returncode:
        failures.append({"returncode": returncode, "command": command})
if failures:
    raise RuntimeError({"inference_failures": failures})
inference_seconds = time.monotonic() - started
print(f"Two-shard inference completed in {inference_seconds / 60.0:.2f} minutes")


In [ ]:
from collections import Counter, defaultdict

import numpy as np
from scipy.spatial import cKDTree
import polars as pl
import tracksdata as td


GAP_RADIUS_GRID = 5.0
GAP_MIN_ACCELERATION_UM = 6.5
GAP_SCALE_UM = np.asarray((1.625, 0.40625, 0.40625), dtype=np.float64)
GAP_GRID_UM = 1.625


def close_internal_track_gaps(graph) -> int:
    node_rows = list(
        graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
        .select("node_id", "t", "z", "y", "x")
        .iter_rows(named=True)
    )
    node_ids = np.asarray([int(row["node_id"]) for row in node_rows])
    times = np.asarray([int(row["t"]) for row in node_rows])
    coords = {
        int(row["node_id"]): np.asarray(
            (row["z"], row["y"], row["x"]), dtype=np.float64
        ) * GAP_SCALE_UM
        for row in node_rows
    }
    edge_rows = list(
        graph.edge_attrs(attr_keys=["source_id", "target_id"])
        .select("source_id", "target_id")
        .iter_rows(named=True)
    )
    predecessor = {
        int(row["target_id"]): int(row["source_id"]) for row in edge_rows
    }
    successor = {
        int(row["source_id"]): int(row["target_id"]) for row in edge_rows
    }
    indegree = Counter(int(row["target_id"]) for row in edge_rows)
    outdegree = Counter(int(row["source_id"]) for row in edge_rows)
    inclusive_radius_um = np.nextafter(GAP_RADIUS_GRID * GAP_GRID_UM, np.inf)
    additions = []

    for target_time in sorted(set(times.tolist())):
        source_ids = np.asarray([
            int(node)
            for node in node_ids[times == target_time - 1]
            if outdegree[int(node)] == 0
        ])
        target_ids = np.asarray([
            int(node)
            for node in node_ids[times == target_time]
            if indegree[int(node)] == 0
        ])
        if len(source_ids) == 0 or len(target_ids) == 0:
            continue
        source_coords = np.asarray([coords[int(node)] for node in source_ids])
        target_coords = np.asarray([coords[int(node)] for node in target_ids])
        distances, source_indices = cKDTree(source_coords).query(
            target_coords,
            k=1,
            distance_upper_bound=inclusive_radius_um,
            workers=-1,
        )
        frame_candidates = []
        for target_index, (distance, source_index) in enumerate(
            zip(distances, source_indices, strict=True)
        ):
            if not np.isfinite(distance) or source_index >= len(source_ids):
                continue
            frame_candidates.append((
                int(source_ids[int(source_index)]),
                int(target_ids[target_index]),
                float(distance),
            ))

        claimed_sources = set()
        for source, target, distance in sorted(
            frame_candidates, key=lambda row: (row[0], row[2], row[1])
        ):
            if source in claimed_sources:
                continue
            claimed_sources.add(source)
            previous = predecessor.get(source)
            following = successor.get(target)
            if previous is None or following is None:
                continue
            vector = coords[target] - coords[source]
            previous_vector = coords[source] - coords[previous]
            following_vector = coords[following] - coords[target]
            min_acceleration = min(
                float(np.linalg.norm(vector - previous_vector)),
                float(np.linalg.norm(following_vector - vector)),
            )
            if min_acceleration > GAP_MIN_ACCELERATION_UM:
                continue
            additions.append({
                "source_id": source,
                "target_id": target,
                "edge_prob": 1.0,
                "edge_dist": distance / GAP_GRID_UM,
            })

    if additions:
        graph.bulk_add_edges(additions)
    return len(additions)


DIV_SCALE_UM = np.asarray((1.625, 0.40625, 0.40625), dtype=np.float64)
DIV_FEATURE_MEAN = np.asarray([2.202441724390983, 11.905854115707049, 12.530247823741115, 5.888479769117546, 9.768692640375301, -0.1158564827485851, 0.1986536428846206, 0.9255220846423952, 0.2930777054607188, -0.07684822100498105, 1.0423211975605249, 1.435286145506068], dtype=np.float64)
DIV_FEATURE_SCALE = np.asarray([1.903330989651284, 2.4559407389543093, 2.8046694511907138, 1.494890787522451, 2.9891283923553686, 0.5657847102440112, 1.2386589875245562, 0.2625470538429257, 0.5500648796748153, 0.579434593739502, 0.22143015841180555, 0.5325751612102582], dtype=np.float64)
DIV_COEFFICIENTS = np.asarray([-0.11447652524963121, 1.7968352108138184, -0.7477483127548737, -0.8527828000255034, -2.3768328955997426, -0.9436208184013919, 0.8690712572601186, 0.15578520308451954, 0.04972761045828509, 1.0321210246647057, -0.0735728543439014, 0.170043844145519], dtype=np.float64)
DIV_INTERCEPT = -8.22818022705616
DIV_THRESHOLD = 0.5668243328192077
DIV_MAX_PARENT_LINKED_UM = 12.0
DIV_MAX_PARENT_CANDIDATE_UM = 15.0
DIV_MAX_SISTER_UM = 20.5
DIV_FRAME_FRACTION_CAP = 0.0076
DIV_GLOBAL_FRACTION_CAP = 0.00375


def div_norm(vector):
    return float(np.linalg.norm(vector))


def div_cosine(left, right):
    denominator = div_norm(left) * div_norm(right)
    return float(np.dot(left, right) / denominator) if denominator > 1e-9 else 0.0


def div_sigmoid(value):
    return float(1.0 / (1.0 + np.exp(-np.clip(value, -40.0, 40.0))))


def add_real_division_repairs(graph):
    node_rows = list(
        graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
        .select("node_id", "t", "z", "y", "x")
        .iter_rows(named=True)
    )
    nodes = {
        int(row["node_id"]): (
            int(row["t"]),
            np.asarray((row["z"], row["y"], row["x"]), dtype=np.float64)
            * DIV_SCALE_UM,
        )
        for row in node_rows
    }
    edge_rows = list(
        graph.edge_attrs(attr_keys=["source_id", "target_id"])
        .select("source_id", "target_id")
        .iter_rows(named=True)
    )
    successors = defaultdict(list)
    predecessors = defaultdict(list)
    for row in edge_rows:
        source = int(row["source_id"])
        target = int(row["target_id"])
        successors[source].append(target)
        predecessors[target].append(source)
    by_time = defaultdict(list)
    for node_id, (time, _) in nodes.items():
        by_time[time].append(node_id)

    proposals = []
    rescue_proposals = []
    candidates_scored = 0
    for time in sorted(by_time):
        source_ids = [
            node_id for node_id in by_time[time]
            if len(successors[node_id]) == 1 and len(predecessors[node_id]) == 1
        ]
        orphan_ids = [
            node_id for node_id in by_time.get(time + 1, [])
            if len(predecessors[node_id]) == 0
        ]
        if not source_ids or not orphan_ids:
            continue
        frame_ids = by_time[time + 1]
        frame_positions = np.stack([nodes[node_id][1] for node_id in frame_ids])
        orphan_positions = np.stack([nodes[node_id][1] for node_id in orphan_ids])
        frame_proposals = []
        for source_id in source_ids:
            linked_id = successors[source_id][0]
            if len(successors[linked_id]) != 1:
                continue
            parent = nodes[source_id][1]
            linked = nodes[linked_id][1]
            parent_linked = div_norm(linked - parent)
            if parent_linked > DIV_MAX_PARENT_LINKED_UM:
                continue
            orphan_distances = np.linalg.norm(orphan_positions - linked, axis=1)
            candidate_id = orphan_ids[int(np.argmin(orphan_distances))]
            if len(successors[candidate_id]) != 1:
                continue
            candidate = nodes[candidate_id][1]
            parent_candidate = div_norm(candidate - parent)
            if parent_candidate > DIV_MAX_PARENT_CANDIDATE_UM:
                continue
            sister = div_norm(candidate - linked)
            if sister > DIV_MAX_SISTER_UM:
                continue
            next_linked = nodes[successors[linked_id][0]][1]
            next_candidate = nodes[successors[candidate_id][0]][1]
            separation_growth = div_norm(next_linked - next_candidate) - sister
            if separation_growth < 0.0:
                continue
            previous_id = predecessors[source_id][0]
            previous_velocity = parent - nodes[previous_id][1]
            positive_distances = np.sort(np.linalg.norm(frame_positions - linked, axis=1))
            positive_distances = positive_distances[positive_distances > 1e-9]
            rank = 1 + int(np.sum(positive_distances < sister - 1e-9))
            local_count = int(
                np.sum(np.linalg.norm(frame_positions - parent, axis=1) <= 12.0)
            )
            features = np.asarray(
                (
                    parent_linked,
                    parent_candidate,
                    sister,
                    div_norm((linked + candidate) * 0.5 - parent),
                    abs(parent_linked - parent_candidate),
                    div_cosine(linked - parent, candidate - parent),
                    separation_growth,
                    1.0,
                    div_cosine(previous_velocity, linked - parent),
                    div_cosine(previous_velocity, candidate - parent),
                    float(rank),
                    float(local_count),
                ),
                dtype=np.float64,
            )
            probability = div_sigmoid(
                DIV_INTERCEPT
                + ((features - DIV_FEATURE_MEAN) / DIV_FEATURE_SCALE)
                @ DIV_COEFFICIENTS
            )
            candidates_scored += 1
            proposal = (probability, source_id, candidate_id, parent_candidate)
            if probability >= DIV_THRESHOLD:
                frame_proposals.append(proposal)
            elif (
                probability >= 0.45
                and parent_candidate <= 8.0
                and sister <= 13.0
                and features[3] <= 3.0
                and features[4] <= 3.5
                and features[5] <= -0.5
                and separation_growth >= 1.0
                and rank <= 2
                and local_count <= 2
            ):
                rescue_proposals.append(proposal)

        frame_cap = max(
            1, int(round(max(1, len(source_ids)) * DIV_FRAME_FRACTION_CAP))
        )
        frame_proposals.sort(reverse=True)
        proposals.extend(frame_proposals[:frame_cap])

    global_cap = max(
        1, int(round(max(1, len(edge_rows)) * DIV_GLOBAL_FRACTION_CAP))
    )
    proposals.sort(reverse=True)
    used_sources = set()
    used_targets = set()
    additions = []
    for probability, source_id, candidate_id, parent_candidate in proposals:
        if len(additions) >= global_cap:
            break
        if source_id in used_sources or candidate_id in used_targets:
            continue
        additions.append(
            {
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": probability,
                "edge_dist": parent_candidate / GAP_GRID_UM,
            }
        )
        used_sources.add(source_id)
        used_targets.add(candidate_id)
        successors[source_id].append(candidate_id)
        predecessors[candidate_id].append(source_id)
    if additions:
        graph.bulk_add_edges(additions)

    rescue_added = 0
    if not any(len(targets) >= 2 for targets in successors.values()):
        for probability, source_id, candidate_id, parent_candidate in sorted(
            rescue_proposals, reverse=True
        ):
            if len(successors[source_id]) != 1 or len(predecessors[candidate_id]) != 0:
                continue
            graph.bulk_add_edges(
                [
                    {
                        "source_id": source_id,
                        "target_id": candidate_id,
                        "edge_prob": probability,
                        "edge_dist": parent_candidate / GAP_GRID_UM,
                    }
                ]
            )
            rescue_added = 1
            break
    return len(additions), rescue_added, candidates_scored


def prediction_dir(method: str) -> Path:
    matches = sorted((REPO_DIR / "predictions").glob(f"*/{method}/split_0"))
    if len(matches) != 1:
        raise RuntimeError({"method": method, "prediction_dirs": [str(path) for path in matches]})
    return matches[0]


merged_dir = REPO_DIR / "predictions" / "model43_merged"
if merged_dir.exists():
    shutil.rmtree(merged_dir)
merged_dir.mkdir(parents=True)
for gpu_index in range(GPU_COUNT):
    source_dir = prediction_dir(f"model43_primary_gpu{gpu_index}")
    for geff_path in source_dir.glob("*.geff"):
        destination = merged_dir / geff_path.name
        if destination.exists():
            raise RuntimeError(f"Duplicate prediction graph: {geff_path.stem}")
        shutil.copytree(geff_path, destination)

geff_paths = sorted(merged_dir.glob("*.geff"))
found_stems = [path.stem for path in geff_paths]
if found_stems != test_stems:
    raise RuntimeError({
        "missing": sorted(set(test_stems) - set(found_stems)),
        "extra": sorted(set(found_stems) - set(test_stems)),
    })

columns = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]
submission_path = WORKING_DIR / "submission.csv"
next_row_id = 0
dataset_receipts = []
with submission_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, lineterminator="\n")
    writer.writeheader()
    for geff_path in geff_paths:
        loaded = td.graph.IndexedRXGraph.from_geff(geff_path)
        graph = loaded[0] if isinstance(loaded, tuple) else loaded
        gap_edges_added = close_internal_track_gaps(graph)
        division_edges_added, rescue_edges_added, division_candidates_scored = (
            add_real_division_repairs(graph)
        )
        node_rows = list(
            graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
            .select("node_id", "t", "z", "y", "x")
            .iter_rows(named=True)
        )
        edge_rows = list(
            graph.edge_attrs(attr_keys=["source_id", "target_id"])
            .select("source_id", "target_id")
            .iter_rows(named=True)
        )
        node_ids = {int(row["node_id"]) for row in node_rows}
        node_times = {int(row["node_id"]): int(row["t"]) for row in node_rows}
        indegree: Counter[int] = Counter()
        outdegree: Counter[int] = Counter()
        edge_pairs: set[tuple[int, int]] = set()
        for row in edge_rows:
            source = int(row["source_id"])
            target = int(row["target_id"])
            if source not in node_ids or target not in node_ids:
                raise RuntimeError(f"{geff_path.stem}: dangling edge {source}->{target}")
            if node_times[target] != node_times[source] + 1:
                raise RuntimeError(f"{geff_path.stem}: nonconsecutive edge {source}->{target}")
            if (source, target) in edge_pairs:
                raise RuntimeError(f"{geff_path.stem}: duplicate edge {source}->{target}")
            edge_pairs.add((source, target))
            outdegree[source] += 1
            indegree[target] += 1
        if max(indegree.values(), default=0) > 1 or max(outdegree.values(), default=0) > 2:
            raise RuntimeError(f"{geff_path.stem}: invalid graph degrees")

        for row in node_rows:
            writer.writerow({
                "id": next_row_id,
                "dataset": geff_path.stem,
                "row_type": "node",
                "node_id": int(row["node_id"]),
                "t": int(row["t"]),
                "z": max(0, int(round(float(row["z"])))),
                "y": max(0, int(round(float(row["y"])))),
                "x": max(0, int(round(float(row["x"])))),
                "source_id": -1,
                "target_id": -1,
            })
            next_row_id += 1
        for row in edge_rows:
            writer.writerow({
                "id": next_row_id,
                "dataset": geff_path.stem,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": int(row["source_id"]),
                "target_id": int(row["target_id"]),
            })
            next_row_id += 1
        dataset_receipts.append({
            "dataset": geff_path.stem,
            "nodes": len(node_rows),
            "edges": len(edge_rows),
            "divisions": sum(value == 2 for value in outdegree.values()),
            "max_indegree": max(indegree.values(), default=0),
            "max_outdegree": max(outdegree.values(), default=0),
            "gap_edges_added": gap_edges_added,
            "division_edges_added": division_edges_added,
            "rescue_edges_added": rescue_edges_added,
            "division_candidates_scored": division_candidates_scored,
        })

with submission_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if reader.fieldnames != columns:
        raise RuntimeError({"bad_header": reader.fieldnames})
    expected_id = 0
    seen_datasets = set()
    for row in reader:
        if int(row["id"]) != expected_id:
            raise RuntimeError(f"Nonconsecutive CSV id at row {expected_id}")
        expected_id += 1
        seen_datasets.add(row["dataset"])
if expected_id != next_row_id or seen_datasets != set(test_stems):
    raise RuntimeError("Final CSV audit failed")

receipt = {
    "status": "complete",
    "model": "model43_gap_real_divisions",
    "ground_truth_accessed": False,
    "test_datasets": len(test_stems),
    "rows": next_row_id,
    "inference_seconds": inference_seconds,
    "gpu_names": GPU_NAMES,
    "config": {
        "det_threshold": DET_THRESHOLD,
        "detection_tta": "four-view xy flips from checksum-pinned predictor",
        "ilp_edge_weight": ILP_EDGE_WEIGHT,
        "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
        "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
        "ilp_division_weight": ILP_DIVISION_WEIGHT,
        "postprocessing": "internal gap closing plus real-label division repair",
        "division_threshold": DIV_THRESHOLD,
        "division_ranker_sha256": "6224077133dc2c94a6d22946e2874fc3f2844f2d35138b2621440d28feeb5953",
        "sparse_rescue": "at most one in fork-free datasets",
        "gap_radius_grid": GAP_RADIUS_GRID,
        "gap_radius_um": GAP_RADIUS_GRID * GAP_GRID_UM,
        "gap_min_acceleration_um": GAP_MIN_ACCELERATION_UM,
        "gap_source_capacity": 1,
    },
    "predictor_sha256": actual_predictor_sha256,
    "weight_sha256": actual_weight_sha256,
    "submission_sha256": sha256_file(submission_path),
    "gap_edges_added": sum(row["gap_edges_added"] for row in dataset_receipts),
    "division_edges_added": sum(row["division_edges_added"] for row in dataset_receipts),
    "rescue_edges_added": sum(row["rescue_edges_added"] for row in dataset_receipts),
    "datasets": dataset_receipts,
}
receipt_path = WORKING_DIR / "model43_runtime_receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n")
print(json.dumps(receipt, indent=2, sort_keys=True))
print(f"Wrote {submission_path} ({submission_path.stat().st_size:,} bytes)")
